In [5]:
import joblib
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

data = pd.read_csv("diabetes.csv")

x = data.drop(columns="Outcome")
y = data["Outcome"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
x_train_scaled = pd.DataFrame(scaler.fit_transform(x_train), columns=x.columns)
x_test_scaled = pd.DataFrame(scaler.transform(x_test), columns=x.columns)

models = {
    "Logistic Regression": LogisticRegression(),
    "SVM": SVC(probability=True, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

rows = []
for name, model in models.items():
    model.fit(x_train_scaled, y_train)
    pred = model.predict(x_test_scaled)
    prob = model.predict_proba(x_test_scaled)[:, 1]
    rows.append(
        {
            "Model": name,
            "Accuracy": accuracy_score(y_test, pred),
            "Precision": precision_score(y_test, pred),
            "Recall": recall_score(y_test, pred),
            "F1": f1_score(y_test, pred),
            "ROC_AUC": roc_auc_score(y_test, prob),
        }
    )

results = pd.DataFrame(rows).sort_values("ROC_AUC", ascending=False)
results.to_csv("results.csv", index=False)
print(results.round(4).to_string(index=False))

best_name = results.iloc[0]["Model"]
print("\nBest model by ROC AUC:", best_name)

final_model = make_pipeline(StandardScaler(), models[best_name])
final_model.fit(x_train, y_train)

joblib.dump(final_model, "model.pkl")
print("Saved model.pkl")

              Model  Accuracy  Precision  Recall     F1  ROC_AUC
      Random Forest    0.7338     0.6250  0.6364 0.6306   0.8177
Logistic Regression    0.7532     0.6491  0.6727 0.6607   0.8147
  Gradient Boosting    0.7403     0.6271  0.6727 0.6491   0.8077
                SVM    0.7338     0.6458  0.5636 0.6019   0.8051
      Decision Tree    0.6948     0.5588  0.6909 0.6179   0.8006

Best model by ROC AUC: Random Forest
Saved model.pkl
